# LAB1 – Bài 6: Tiền xử lý dữ liệu thiếu và loại bỏ thuộc tính không cần

**Họ và tên:** TRẦN TRUNG THÔNG  
**MSSV:** 6451071075  
**Môn học:** Khai thác dữ liệu  
**Nguồn bài tập:** Tài liệu LAB1, mục IV.2 – Yêu cầu lập trình (trang 15–27 theo trình đọc PDF).  
**Dữ liệu:** `titanic.csv` do giảng viên cung cấp (1.309 dòng, 14 thuộc tính).

> Mỗi bài là một notebook độc lập. Upload `titanic.csv` vào Colab, chạy từ trên xuống, bổ sung nhận xét theo kết quả thực tế và lưu notebook sau khi chạy.


In [ ]:
# Thông tin sinh viên (in bằng lệnh print theo yêu cầu của giảng viên)
HO_TEN = "TRẦN TRUNG THÔNG"
MSSV = "6451071075"
print("Họ và tên:", HO_TEN)
print("MSSV:", MSSV)
print("LAB1 - Bài 6: Tiền xử lý dữ liệu thiếu và loại bỏ thuộc tính không cần")


## 1. Đọc dữ liệu và chuẩn bị thư viện

Tải `titanic.csv` vào bảng **Tệp** của Google Colab trước khi chạy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)

# Tải file titanic.csv của giảng viên lên Colab trước khi chạy.
# Có thể tải tại /content/titanic.csv hoặc /titanic.csv như phiên làm việc trước.
paths = ['/content/titanic.csv', '/titanic.csv', 'titanic.csv', 'titanic3.csv']
titanic_path = next((Path(p) for p in paths if Path(p).is_file()), None)
if titanic_path is None:
    raise FileNotFoundError('Không tìm thấy titanic.csv. Vào thanh Tệp của Colab và tải file lên trước.')

df = pd.read_csv(titanic_path)
df.columns = df.columns.str.strip().str.lower()
df = df.rename(columns={'survival': 'survived'})
for col in ['pclass', 'survived', 'age', 'sibsp', 'parch', 'fare']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
print('Đã đọc:', titanic_path)
print('Kích thước dữ liệu:', df.shape)
assert df.shape == (1309, 14), 'Hãy kiểm tra lại file Titanic do giảng viên cung cấp'


## Câu 6 — Tiền xử lý thiếu dữ liệu và xóa cột không cần thiết

Tài liệu gợi ý:
- Tách `deck` (boong tàu) từ chữ đầu của `cabin` trước khi xóa `cabin`.
- Xóa `body`, `boat`, `home.dest`, `ticket`, `cabin`. `boat`/`body` còn có nguy cơ **rò rỉ thông tin kết quả sống sót** nếu dùng huấn luyện mô hình.
- Điền tuổi thiếu và `embarked` thiếu. PDF minh họa điền tuổi bằng số ngẫu nhiên theo trung bình, độ lệch chuẩn; ở notebook này **mặc định điền trung vị** (ít ngẫu nhiên, dễ kiểm tra), và có một ô *tùy chọn* phía dưới cho cách ngẫu nhiên trong tài liệu.

**Chỉ thao tác trên bản sao `df_clean`.**

## 2. Thực hiện bài toán

In [ ]:
df_rel = df.copy()
df_rel['relatives'] = df_rel['sibsp'].fillna(0) + df_rel['parch'].fillna(0)
df_rel['not_alone'] = (df_rel['relatives'] > 0).astype(int)

In [ ]:
df_clean = df_rel.copy()
# Tách chữ đầu tiên trong mã cabin: ví dụ C123 -> C; không có cabin -> U.
if 'cabin' in df_clean:
    df_clean['deck'] = df_clean['cabin'].astype('string').str.extract(r'^([A-Za-z])', expand=False).fillna('U')
else:
    df_clean['deck'] = 'U'

# Loại bỏ cột không cần thiết nếu chúng có trong bộ dữ liệu.
df_clean = df_clean.drop(columns=['ticket','boat','body','home.dest','cabin'], errors='ignore')

# Điền thiếu
if df_clean['age'].notna().any():
    df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())
else:
    raise ValueError('Cột age không có giá trị hợp lệ để tính trung vị')
if df_clean['embarked'].notna().any():
    df_clean['embarked'] = df_clean['embarked'].fillna(df_clean['embarked'].mode()[0])
else:
    df_clean['embarked'] = 'Missing'
df_clean['fare'] = df_clean['fare'].fillna(df_clean['fare'].median())

print('Các cột còn lại:', df_clean.columns.tolist())
print('Số ô thiếu còn lại trên từng cột:')
display(df_clean.isna().sum().sort_values(ascending=False).head(12))
display(df_clean.head())

## 3. Nhận xét kết quả

**Bạn viết nhận xét dựa trên số liệu và biểu đồ thực tế ở trên.** Không kết luận từ các nhóm có rất ít quan sát.

**Chụp màn hình:** Đảm bảo ảnh chụp thấy code, kết quả và **thanh taskbar Windows** ở dưới cùng. Nên chụp nhiều ảnh nếu kết quả dài.